# Gradient_Descent - 경사하강법

In [ ]:
import sklearn

print(sklearn.__version__)

1.6.1


In [ ]:
from sklearn.datasets import load_diabetes
import pandas as pd
import numpy as np

diabetes = load_diabetes()
diabetesDF = pd.DataFrame(diabetes.data, columns=diabetes.feature_names)
diabetesDF['target'] = diabetes.target
print(diabetesDF.shape)
diabetesDF.head()

(442, 11)


,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6,target
0,0.038076,0.050680,0.061696,0.021872,-0.044223,-0.034821,-0.043401,-0.002592,0.019907,-0.017646,151.0
1,-0.001882,-0.044642,-0.051474,-0.026328,-0.008449,-0.019163,0.074412,-0.039493,-0.068332,-0.092204,75.0
2,0.085299,0.050680,0.044451,-0.005670,-0.045599,-0.034194,-0.032356,-0.002592,0.002861,-0.025930,141.0
3,-0.089063,-0.044642,-0.011595,-0.036656,0.012191,0.024991,-0.036038,0.034309,0.022688,-0.009362,206.0
4,0.005383,-0.044642,-0.036385,0.021872,0.003935,0.015596,0.008142,-0.002592,-0.031988,-0.046641,135.0


## 가중치와 편향의 업데이트 값을 계산하는 함수
## **parameter**
- w1 : age 변수(피처)의 weight값
- w2 : sex 변수(피처)의 weight값
- w3 : bmi 변수(피처)의 weight값
- w4 : bp 변수(피처)의 weight값
- bias
- N : input data 건수

In [ ]:
# 기존 강의방식 유지(파라미터가 많아 코드가 길어짐)

def get_update_weights_values(bias, w1, w2, w3, w4, age, sex, bmi, bp, target, learning_rate=0.01):

    N = len(target)  # 타겟 데이터 개수

    # 1. 예측값 계산
    predicted = w1 * age + w2 * sex + w3 * bmi + w4 * bp + bias
    diff = target - predicted

    # 2. 오차 계산
    bias_factors = np.ones((N,))

    # 3. 가중치 및 편향 업데이트량 계산 (기여도/기울기)
    w1_update = -(2/N) * learning_rate * np.dot(age.T, diff)
    w2_update = -(2/N) * learning_rate * np.dot(sex.T, diff)
    w3_update = -(2/N) * learning_rate * np.dot(bmi.T, diff)
    w4_update = -(2/N) * learning_rate * np.dot(bp.T, diff)
    bias_update = -(2/N) * learning_rate * np.dot(bias_factors.T, diff)

    # 4. 손실 함수 (MSE) 계산
    mse_loss = np.mean(np.square(diff))

    # weight와 bias가 update되어야 할 값과 MSE 값을 반환
    return w1_update, w2_update, w3_update, w4_update, bias_update, mse_loss

In [ ]:
'''
# 권장하는 방식(실제 딥러닝 프레임워크 작동방식)

def get_update_weights_value_matrix(bias, weights, X, target, learning_rate=0.01):
    """
    X: (N, 4) 형태의 입력 데이터 Matrix (age, sex, bmi, bp)
    weights: (4,) 형태의 가중치 Vector
    """
    N = len(target)

    # 1. 예측값 계산 (행렬 곱)
    predicted = np.dot(X, weights) + bias

    # 2. 오차 계산
    diff = target - predicted

    # 3. 가중치 및 편향 업데이트량 일괄 계산
    # X.T (4, N) dot diff (N,) -> w_update (4,)
    w_update = -(2/N) * learning_rate * np.dot(X.T, diff)
    bias_update = -(2/N) * learning_rate * np.sum(diff)

    # 4. 손실 함수 (MSE) 계산
    mse_loss = np.mean(np.square(diff))

    return bias_update, w_update, mse_loss
'''

'\n# 권장하는 방식(실제 딥러닝 프레임워크 작동방식)\n\ndef get_update_weights_value_matrix(bias, weights, X, target, learning_rate=0.01):\n    """\n    X: (N, 4) 형태의 입력 데이터 Matrix (age, sex, bmi, bp)\n    weights: (4,) 형태의 가중치 Vector\n    """\n    N = len(target)\n\n    # 1. 예측값 계산 (행렬 곱)\n    predicted = np.dot(X, weights) + bias\n\n    # 2. 오차 계산\n    diff = target - predicted\n\n    # 3. 가중치 및 편향 업데이트량 일괄 계산\n    # X.T (4, N) dot diff (N,) -> w_update (4,)\n    w_update = -(2/N) * learning_rate * np.dot(X.T, diff)\n    bias_update = -(2/N) * learning_rate * np.sum(diff)\n\n    # 4. 손실 함수 (MSE) 계산\n    mse_loss = np.mean(np.square(diff))\n\n    return bias_update, w_update, mse_loss\n'

# Gradient Descent를 적용하는 함수 생성
- iter_epochs 수만큼 반복적으로 get_update_weights_value()를 호출하여, <br>
update될 weihgt/bias값을 구한 뒤
weight/bias를 update 적용

In [ ]:
def gradient_descent(features, target, iter_epochs=1000, verbose=True):

    w1 = np.zeros((1,))
    w2 = np.zeros((1,))
    w3 = np.zeros((1,))
    w4 = np.zeros((1,))
    bias = np.zeros((1,))
    print('최초 w1, w2, w3, w4, bias:', w1, w2, w3, w4, bias)

    learning_rate = 0.01
    age = features[:, 0]
    sex = features[:, 1]
    bmi = features[:, 2]
    bp = features[:, 3]

    for i in range(iter_epochs):

        bias_update, w1_update, w2_update, w3_update, w4_update, mse_loss = get_update_weights_values(bias, w1, w2, w3, w4, age, sex, bmi, bp, target, learning_rate)

        w1 = w1 - w1_update
        w2 = w2 - w2_update
        w3 = w3 - w3_update
        w4 = w4 - w4_update
        bias = bias - bias_update
        if verbose:
            print('Epoch:', i+1, '/', iter_epochs)
            print('w1:', w1, 'w2:', w2, 'w3:', w3, 'w4:', w4, 'bias:', bias)

        return w1, w2, w3, w4, bias

# 학습 팁 & 주의사항

- 학습률(Learning Rate) 조정: 사이킷런의 당뇨병 데이터셋은 피처들이 이미 **정규화(Mean=0, Std=1 내외)** 되어 있어 기본 0.01~0.1 학습률로도 잘 발산하지 않고 실습하기 수월합니다.

- 타겟값 스케일: target 값의 범위가 25-346 정도로 크기 때문에, 초반 **loss(MSE)** 값이 수천~수만 단위로 크게 출력되는 것은 정상입니다.

### Gradient Descent 적용
* 신경망은 데이터를 정규화/표준화 작업을 미리 선행해 주어야 함.
* 이를 위해 사이킷런의 MinMaxScaler를 이용하여 개별 feature값은 0~1사이 값으로 변환후 학습 적용.

In [ ]:
'''
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
scaled_features = scaler.fit_transform(bostonDF[['RM', 'LSTAT']])
'''

features = diabetesDF[['age', 'sex', 'bmi', 'bp']].values
target = diabetesDF['target'].values

w1, w2, w3, w4, bias = gradient_descent(features, diabetesDF['target'].values, iter_epochs=5000, verbose=True)
print('##### 최종 w1, w2, w3, w4, bias #######')
print(w1, w2, w3, w4, bias)


최초 w1, w2, w3, w4, bias: [0.] [0.] [0.] [0.] [0.]
Epoch: 1 / 5000
w1: [0.00315454] w2: [0.04296087] w3: [0.0323411] w4: [3.04266968] bias: [0.01376394]
##### 최종 w1, w2, w3, w4, bias #######
[0.00315454] [0.04296087] [0.0323411] [3.04266968] [0.01376394]


### 계산된 Weight와 Bias를 이용하여 Price 예측
* 예측 feature 역시 0~1사이의 scaled값을 이용하고 Weight와 bias를 적용하여 예측값 계산.

In [ ]:
'''
predicted = scaled_features[:, 0]*w1 + scaled_features[:, 1]*w2 + bias
bostonDF['PREDICTED_PRICE'] = predicted
bostonDF.head(10)
'''
predicted = features[:, 0]*w1 + features[:, 1]*w2 + features[:, 2]*w3 + features[:, 3]*w4 + bias
diabetesDF['PREDICTED_TARGET'] = predicted
diabetesDF.head(10)

,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6,target,PREDICTED_TARGET
0,0.038076,0.050680,0.061696,0.021872,-0.044223,-0.034821,-0.043401,-0.002592,0.019907,-0.017646,151.0,0.084607
1,-0.001882,-0.044642,-0.051474,-0.026328,-0.008449,-0.019163,0.074412,-0.039493,-0.068332,-0.092204,75.0,-0.069931
2,0.085299,0.050680,0.044451,-0.005670,-0.045599,-0.034194,-0.032356,-0.002592,0.002861,-0.025930,141.0,0.000395
3,-0.089063,-0.044642,-0.011595,-0.036656,0.012191,0.024991,-0.036038,0.034309,0.022688,-0.009362,206.0,-0.100342
4,0.005383,-0.044642,-0.036385,0.021872,0.003935,0.015596,0.008142,-0.002592,-0.031988,-0.046641,135.0,0.077237
5,-0.092695,-0.044642,-0.040696,-0.019442,-0.068991,-0.079288,0.041277,-0.076395,-0.041176,-0.096346,97.0,-0.048918
6,-0.045472,0.050680,-0.047163,-0.015999,-0.040096,-0.024800,0.000779,-0.039493,-0.062917,-0.038357,138.0,-0.034407
7,0.063504,0.050680,-0.001895,0.066629,0.090620,0.108914,0.022869,0.017703,-0.035816,0.003064,63.0,0.218812
8,0.041708,0.050680,0.061696,-0.040099,-0.013953,0.006202,-0.028674,-0.002592,-0.014960,0.011349,110.0,-0.103940
9,-0.070900,-0.044642,0.039062,-0.033213,-0.012577,-0.034508,-0.024993,-0.002592,0.067737,-0.013504,310.0,-0.088171


### Keras를 이용하여 모델 학습 및 예측
* Dense Layer를 이용하여 퍼셉트론 구현. units는 1로 설정.

In [ ]:
from tensorflow.keras.layers import Dense
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam

model = Sequential([
    # 단 하나의 units 설정. input_shape는 2차원, 회귀이므로 activation은 설정하지 않음.
    # weight와 bias 초기화는 kernel_inbitializer와 bias_initializer를 이용.
    Dense(1, input_shape=(4, ), activation=None, kernel_initializer='zeros', bias_initializer='ones')
])
# Adam optimizer를 이용하고 Loss 함수는 Mean Squared Error, 성능 측정 역시 MSE를 이용하여 학습 수행.
model.compile(optimizer=Adam(learning_rate=0.01), loss='mse', metrics=['mse'])
model.fit(features, diabetesDF['target'].values, epochs=1000)


/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/1000
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - loss: 28751.0332 - mse: 28751.0332
Epoch 2/1000
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 28708.0703 - mse: 28708.0703 
Epoch 3/1000
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 28664.8027 - mse: 28664.8027 
Epoch 4/1000
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 28622.1875 - mse: 28622.1875 
Epoch 5/1000
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 28578.9160 - mse: 28578.9160 
Epoch 6/1000
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 28536.5703 - mse: 28536.5703 
Epoch 7/1000
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 28493.4141 - mse: 28493.4141 
Epoch 8/1000
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 28450.7852 - mse: 28450.7852 
Epoch 9/1000
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 28408.3965 - mse: 28408.3965 
Epoch 10/1000
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 28365.7559 - mse: 28365.7559 
Epoch 11/1000
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 28322.9180 - mse: 28322.9180 
Epoch 12

### Keras로 학습된 모델을 이용하여 예측 수행.

In [ ]:
predicted = model.predict(features)
diabetesDF['KERAS_PREDICTED_PRICE'] = predicted
diabetesDF.head(10)

14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step


,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6,target,PREDICTED_TARGET,KERAS_PREDICTED_PRICE
0,0.038076,0.050680,0.061696,0.021872,-0.044223,-0.034821,-0.043401,-0.002592,0.019907,-0.017646,151.0,0.084607,137.598053
1,-0.001882,-0.044642,-0.051474,-0.026328,-0.008449,-0.019163,0.074412,-0.039493,-0.068332,-0.092204,75.0,-0.069931,114.894592
2,0.085299,0.050680,0.044451,-0.005670,-0.045599,-0.034194,-0.032356,-0.002592,0.002861,-0.025930,141.0,0.000395,135.739395
3,-0.089063,-0.044642,-0.011595,-0.036656,0.012191,0.024991,-0.036038,0.034309,0.022688,-0.009362,206.0,-0.100342,112.496986
4,0.005383,-0.044642,-0.036385,0.021872,0.003935,0.015596,0.008142,-0.002592,-0.031988,-0.046641,135.0,0.077237,122.541206
5,-0.092695,-0.044642,-0.040696,-0.019442,-0.068991,-0.079288,0.041277,-0.076395,-0.041176,-0.096346,97.0,-0.048918,110.688210
6,-0.045472,0.050680,-0.047163,-0.015999,-0.040096,-0.024800,0.000779,-0.039493,-0.062917,-0.038357,138.0,-0.034407,114.666969
7,0.063504,0.050680,-0.001895,0.066629,0.090620,0.108914,0.022869,0.017703,-0.035816,0.003064,63.0,0.218812,136.735916
8,0.041708,0.050680,0.061696,-0.040099,-0.013953,0.006202,-0.028674,-0.002592,-0.014960,0.011349,110.0,-0.103940,130.972580
9,-0.070900,-0.044642,0.039062,-0.033213,-0.012577,-0.034508,-0.024993,-0.002592,0.067737,-0.013504,310.0,-0.088171,120.167702


# Stochastic Gradient Descent와 Mini Batch Gradient Descent 구현
* SGD 는 전체 데이터에서 한건만 임의로 선택하여 Gradient Descent 로 Weight/Bias Update 계산한 뒤 Weight/Bias 적용
* Mini Batch GD는 전체 데이터에서 Batch 건수만큼 데이터를 선택하여 Gradient Descent로 Weight/Bias Update 계산한 뒤 Weight/Bias 적용

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_diabetes

diabetes = load_diabetes()
diabetesDF = pd.DataFrame(diabetes.data, columns=diabetes.feature_names)
diabetesDF['target'] = diabetes.target
print(diabetesDF.shape)
diabetesDF.head()

(442, 11)


,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6,target
0,0.038076,0.050680,0.061696,0.021872,-0.044223,-0.034821,-0.043401,-0.002592,0.019907,-0.017646,151.0
1,-0.001882,-0.044642,-0.051474,-0.026328,-0.008449,-0.019163,0.074412,-0.039493,-0.068332,-0.092204,75.0
2,0.085299,0.050680,0.044451,-0.005670,-0.045599,-0.034194,-0.032356,-0.002592,0.002861,-0.025930,141.0
3,-0.089063,-0.044642,-0.011595,-0.036656,0.012191,0.024991,-0.036038,0.034309,0.022688,-0.009362,206.0
4,0.005383,-0.044642,-0.036385,0.021872,0.003935,0.015596,0.008142,-0.002592,-0.031988,-0.046641,135.0


# SGD 기반으로 Weight/Bias update 값 구하기

In [ ]:
def get_update_weights_values_sgd(bias, w1, w2, w3, w4, age_sgd, sex_sgd, bmi_sgd, bp_sgd, target_sgd, learning_rate=0.01):

    # 타겟 데이터 개수
    N = target_sgd.shape[0]

    predicted_sgd = w1 * age_sgd + w2 * sex_sgd + w3 * bmi_sgd + w4 * bp_sgd + bias

    diff_sgd = target_sgd - predicted_sgd

    bias_factors = np.ones((N,))

    w1_update = -(2/N) * learning_rate * np.dot(age_sgd.T, diff_sgd)
    w2_update = -(2/N) * learning_rate * np.dot(sex_sgd.T, diff_sgd)
    w3_update = -(2/N) * learning_rate * np.dot(bmi_sgd.T, diff_sgd)
    w4_update = -(2/N) * learning_rate * np.dot(bp_sgd.T, diff_sgd)
    bias_update = -(2/N) * learning_rate * np.dot(bias_factors.T, diff_sgd)

    # Mean Squared Error값을 계산
    # mse_loss = np.mean(np.square(diff_sgd))

    # weight와 bias가 update 되어야 할 값들을 반환
    return bias_update, w1_update, w2_update, w3_update, w4_update

# SGD 수행하기(Stochastic Gradient Boosting)

In [ ]:
print(diabetesDF['target'].values.shape)
print(np.random.choice(diabetesDF['target'].values.shape[0], 1))
print(np.random.choice(506, 1))

(442,)
[413]
[213]


In [ ]:
def stoch_gradient_descent(features, target, iter_epochs=1000, verbose=True):
    np.random.seed = 2026
    w1 = np.zeros((1,))
    w2 = np.zeros((1,))
    w3 = np.zeros((1,))
    w4 = np.zeros((1,))
    bias = np.zeros((1,))
    print('initial w1, w2, w3, w4, bias:', w1, w2, w3, w4, bias)

    # learning_rate와 features(age, sex, bmi, bp) 지정.
    learning_rate = 0.01
    age = features[:, 0]
    sex = features[:, 1]
    bmi = features[:, 2]
    bp = features[:, 3]


    for i in range(iter_epochs):
        # iteration 시마다 stoch_gradient_descent를 수행할 데이터를 한개만 추출,
        stochastic_idx = np.random.choice(target.shape[0], 1)
        age_sgd = age[stochastic_idx]
        sex_sgd = sex[stochastic_idx]
        bmi_sgd = bmi[stochastic_idx]
        bp_sgd = bp[stochastic_idx]
        target_sgd = target[stochastic_idx]

        # SGD로 구한 weight/biasd의 update 구하기.
        bias_update, w1_update, w2_update, w3_update, w4_update = get_update_weights_values_sgd(bias, w1, w2, w3, w4, age_sgd, sex_sgd, bmi_sgd, bp_sgd, target_sgd, learning_rate)

        # SGD로 구한 weight/biasd의 update 적용.
        w1 = w1 - w1_update
        w2 = w2 - w2_update
        w3 = w3 - w3_update
        w4 = w4 - w4_update
        bias = bias - bias_update

        if verbose:
            print('Epoch:', i+1, '/', iter_epochs)
            # Loss (전체 학습 data 기반)
            predicted = w1 * age + w2 * sex + w3 * bmi + w4 * bp + bias
            diff = target - predicted
            mse_loss = np.mean(np.square(diff))
            print('w1:', w1, 'w2:', w2, 'w3:', w3, 'w4:', w4, 'bias:', bias)

    return w1, w2, w3, w4, bias


In [ ]:
'''
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
scaled_features = scaler.fit_transform(bostonDF[['RM', 'LSTAT']])

w1, w2, bias = st_gradient_descent(scaled_features, bostonDF['PRICE'].values, iter_epochs=5000, verbose=True)
print('##### 최종 w1, w2, bias #######')
print(w1, w2, bias)
'''
features = diabetesDF[['age', 'sex', 'bmi', 'bp']].values
target = diabetesDF['target'].values

w1, w2, w3, w4, bias = stoch_gradient_descent(features, diabetesDF['target'].values, iter_epochs=5000, verbose=True)
print('##### 최종 w1, w2, w3, w4, bias #######')
print(w1, w2, w3, w4, bias)


스트리밍 출력 내용이 길어서 마지막 5000줄이 삭제되었습니다.
Epoch: 2502 / 5000
w1: [27.24134929] w2: [4.11691101] w3: [100.23798206] w4: [73.77533872] bias: [161.54194491]
Epoch: 2503 / 5000
w1: [27.23871468] w2: [4.1387598] w3: [100.25578969] w4: [73.7646338] bias: [161.0525186]
Epoch: 2504 / 5000
w1: [27.25886455] w2: [4.16025895] w3: [100.27954116] w4: [73.78228715] bias: [160.5709244]
Epoch: 2505 / 5000
w1: [27.25091713] w2: [4.16789632] w3: [100.27681928] w4: [73.79440336] bias: [160.72162196]
Epoch: 2506 / 5000
w1: [27.25373636] w2: [4.23476877] w3: [100.37653021] w4: [73.7925829] bias: [159.22363825]
Epoch: 2507 / 5000
w1: [27.41176462] w2: [4.35406192] w3: [100.36192236] w4: [73.94131436] bias: [161.57748346]
Epoch: 2508 / 5000
w1: [27.36539908] w2: [4.32651396] w3: [100.33776021] w4: [73.94439661] bias: [161.03391802]
Epoch: 2509 / 5000
w1: [27.33153773] w2: [4.35738015] w3: [100.29584735] w4: [73.96021957] bias: [160.34249651]
Epoch: 2510 / 5000
w1: [27.31928316] w2: [4.28849253] w3: [100.34969881] 

In [ ]:
predicted = features[:, 0]*w1 + features[:, 1]*w2 + features[:, 2]*w3 + features[:, 3]*w4 + bias
diabetesDF['SGD_PREDICTED_TARGET'] = predicted
diabetesDF.head(10)

,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6,target,SGD_PREDICTED_TARGET
0,0.038076,0.050680,0.061696,0.021872,-0.044223,-0.034821,-0.043401,-0.002592,0.019907,-0.017646,151.0,172.009121
1,-0.001882,-0.044642,-0.051474,-0.026328,-0.008449,-0.019163,0.074412,-0.039493,-0.068332,-0.092204,75.0,141.854375
2,0.085299,0.050680,0.044451,-0.005670,-0.045599,-0.034194,-0.032356,-0.002592,0.002861,-0.025930,141.0,167.584729
3,-0.089063,-0.044642,-0.011595,-0.036656,0.012191,0.024991,-0.036038,0.034309,0.022688,-0.009362,206.0,143.413399
4,0.005383,-0.044642,-0.036385,0.021872,0.003935,0.015596,0.008142,-0.002592,-0.031988,-0.046641,135.0,151.382886
5,-0.092695,-0.044642,-0.040696,-0.019442,-0.068991,-0.079288,0.041277,-0.076395,-0.041176,-0.096346,97.0,140.137998
6,-0.045472,0.050680,-0.047163,-0.015999,-0.040096,-0.024800,0.000779,-0.039493,-0.062917,-0.038357,138.0,142.673419
7,0.063504,0.050680,-0.001895,0.066629,0.090620,0.108914,0.022869,0.017703,-0.035816,0.003064,63.0,167.492054
8,0.041708,0.050680,0.061696,-0.040099,-0.013953,0.006202,-0.028674,-0.002592,-0.014960,0.011349,110.0,163.996115
9,-0.070900,-0.044642,0.039062,-0.033213,-0.012577,-0.034508,-0.024993,-0.002592,0.067737,-0.013504,310.0,154.136019


### iteration시마다 **일정한 batch 크기**만큼의 데이터를 <br> **random**하게 가져와서 GD를 수행하는 **Mini-Batch GD** 수행

In [ ]:
def get_update_weights_values_batch(bias, w1, w2, w3, w4, age_batch, sex_batch, bmi_batch, bp_batch, target_batch, learning_rate):

    # 데이터 개수
    N = target_batch.shape[0]
    # 예측값
    predicted_batch = w1 * age_batch + w2 * sex_batch + w3 * bmi_batch + w4 * bp_batch + bias
    # 실제값 - 예측값
    diff_batch = target_batch - predicted_batch
    # array 기반 bia 설정
    bias_factors = np.ones((N,))

    # weight와 bias를 얼마나 update할 것인지를 계산.
    w1_update = -(2/N) * learning_rate * np.dot(age_batch.T, diff_batch)
    w2_update = -(2/N) * learning_rate * np.dot(sex_batch.T, diff_batch)
    w3_update = -(2/N) * learning_rate * np.dot(bmi_batch.T, diff_batch)
    w4_update = -(2/N) * learning_rate * np.dot(bp_batch.T, diff_batch)
    bias_update = -(2/N) * learning_rate * np.dot(bias_factors.T, diff_batch)

    # Mean Squared Error값을 계산.
    #mse_loss = np.mean(np.square(diff))

    return bias_update, w1_update, w2_update, w3_update, w4_update


In [ ]:
batch_idxes = np.random.choice(442, 30)
print(batch_idxes)

diabetesDF['bmi'].values[batch_idxes]

[365 322 157 163 345 160  69 160 141 343  75 271 182  42  44 219 166 435
 217 432 167 411 169 416 391 420 215 131 313 376]


array([-0.03854032,  0.06169621, -0.03315126,  0.07247433, -0.00297252,
       -0.06225218, -0.046085  , -0.06225218,  0.04768465,  0.01858372,
       -0.03099563,  0.00888341,  0.00564998, -0.0105172 ,  0.06816308,
       -0.04177375, -0.06009656, -0.02345095,  0.03151747,  0.05522933,
        0.06924089,  0.05846277, -0.02668438,  0.08001901, -0.06979687,
       -0.03638469,  0.09403057, -0.06979687,  0.05307371,  0.06816308])

In [ ]:
def batch_random_gradient_descent(features, target, iter_epochs=1000, batch_size=30, verbose=True):
    np.random.seed = 2026
    w1 = np.zeros((1,))
    w2 = np.zeros((1,))
    w3 = np.zeros((1,))
    w4 = np.zeros((1,))
    bias = np.zeros((1,))
    print('initial w1, w2, w3, w4, bias:', w1, w2, w3, w4, bias)

    learning_rate = 0.01
    age = features[:, 0]
    sex = features[:, 1]
    bmi = features[:, 2]
    bp = features[:, 3]

    for i in range(iter_epochs):
        # batch_size 갯수만큼 데이터를 임의로 선택.
        batch_idxes = np.random.choice(target.shape[0], batch_size)
        age_batch = age[batch_idxes]
        sex_batch = sex[batch_idxes]
        bmi_batch = bmi[batch_idxes]
        bp_batch = bp[batch_idxes]
        target_batch = target[batch_idxes]
        # Batch GD 기반으로 Weight/Bias의 Update를 구함.
        bias_update, w1_update, w2_update, w3_update, w4_update = get_update_weights_values_batch(bias, w1, w2, w3, w4, age_batch, sex_batch, bmi_batch, bp_batch, target_batch, learning_rate)

        # Batch GD로 구한 weight/bias의 update 적용.
        w1 = w1 - w1_update
        w2 = w2 - w2_update
        w3 = w3 - w3_update
        w4 = w4 - w4_update
        bias = bias - bias_update
        if verbose:
            print('Epoch:', i+1, '/', iter_epochs)
            # 전체 학습 데이터 기반 Loss
            predicted = w1 * age + w2 * sex + w3 * bmi + w4 * bp + bias
            diff = target - predicted
            mse_loss = np.mean(np.square(diff))
            print('w1:', w1, 'w2:', w2, 'w3:', w3, 'w4:', w4, 'bias:', bias)

    return w1, w2, w3, w4, bias

In [ ]:
w1, w2, w3, w4, bias = batch_random_gradient_descent(features, diabetesDF['target'].values, iter_epochs=5000, batch_size=30, verbose=True)
print('##### 최종 w1, w2, w3, w4, bias #######')
print(w1, w2, w3, w4, bias)

스트리밍 출력 내용이 길어서 마지막 5000줄이 삭제되었습니다.
Epoch: 2502 / 5000
w1: [29.99767099] w2: [6.58698999] w3: [99.57716439] w4: [73.17925284] bias: [153.34608398]
Epoch: 2503 / 5000
w1: [30.01430714] w2: [6.57749608] w3: [99.62511641] w4: [73.20642596] bias: [153.55976079]
Epoch: 2504 / 5000
w1: [30.01347816] w2: [6.57887263] w3: [99.6661726] w4: [73.24562248] bias: [153.46332831]
Epoch: 2505 / 5000
w1: [30.01462607] w2: [6.56855004] w3: [99.71473448] w4: [73.25597499] bias: [153.79014211]
Epoch: 2506 / 5000
w1: [30.01468248] w2: [6.55358557] w3: [99.74358922] w4: [73.25918407] bias: [153.73701304]
Epoch: 2507 / 5000
w1: [30.03966132] w2: [6.55283855] w3: [99.76884403] w4: [73.29262744] bias: [153.65961701]
Epoch: 2508 / 5000
w1: [30.04405605] w2: [6.54685026] w3: [99.78624067] w4: [73.29795554] bias: [153.3596099]
Epoch: 2509 / 5000
w1: [30.05536589] w2: [6.54301993] w3: [99.81354223] w4: [73.32249764] bias: [153.32539048]
Epoch: 2510 / 5000
w1: [30.0691703] w2: [6.54210696] w3: [99.83780524] w4: [73

In [ ]:
predicted = features[:, 0]*w1 + features[:, 1]*w2 + features[:, 2]*w3 + features[:, 3]*w4 + bias
diabetesDF['BATCH_PREDICTED_TARGET'] = predicted
diabetesDF.head(10)

,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6,target,BATCH_PREDICTED_TARGET
0,0.038076,0.050680,0.061696,0.021872,-0.044223,-0.034821,-0.043401,-0.002592,0.019907,-0.017646,151.0,168.630958
1,-0.001882,-0.044642,-0.051474,-0.026328,-0.008449,-0.019163,0.074412,-0.039493,-0.068332,-0.092204,75.0,138.382023
2,0.085299,0.050680,0.044451,-0.005670,-0.045599,-0.034194,-0.032356,-0.002592,0.002861,-0.025930,141.0,164.258531
3,-0.089063,-0.044642,-0.011595,-0.036656,0.012191,0.024991,-0.036038,0.034309,0.022688,-0.009362,206.0,139.716710
4,0.005383,-0.044642,-0.036385,0.021872,0.003935,0.015596,0.008142,-0.002592,-0.031988,-0.046641,135.0,148.043409
5,-0.092695,-0.044642,-0.040696,-0.019442,-0.068991,-0.079288,0.041277,-0.076395,-0.041176,-0.096346,97.0,136.451885
6,-0.045472,0.050680,-0.047163,-0.015999,-0.040096,-0.024800,0.000779,-0.039493,-0.062917,-0.038357,138.0,138.930860
7,0.063504,0.050680,-0.001895,0.066629,0.090620,0.108914,0.022869,0.017703,-0.035816,0.003064,63.0,164.238981
8,0.041708,0.050680,0.061696,-0.040099,-0.013953,0.006202,-0.028674,-0.002592,-0.014960,0.011349,110.0,160.492391
9,-0.070900,-0.044642,0.039062,-0.033213,-0.012577,-0.034508,-0.024993,-0.002592,0.067737,-0.013504,310.0,150.524353


### iteration 시에 순차적으로 일정한 batch 크기만큼의 데이터를<br> 전체 학습데이터에 걸쳐서 가져오는 Mini-Batch GD 수행

In [ ]:
for batch_step in range(0, 442, 30):
    print(batch_step)

0
30
60
90
120
150
180
210
240
270
300
330
360
390
420


In [ ]:
diabetesDF['target'].values[420:450]

array([146., 212., 233.,  91., 111., 152., 120.,  67., 310.,  94., 183.,
        66., 173.,  72.,  49.,  64.,  48., 178., 104., 132., 220.,  57.])

In [ ]:
def batch_gradient_descent(features, target, iter_epochs=1000, batch_size=30, verbose=True):
    np.random.seed = 2026
    w1 = np.zeros((1,))
    w2 = np.zeros((1,))
    w3 = np.zeros((1,))
    w4 = np.zeros((1,))
    bias = np.zeros((1,))
    print('initial w1, w2, w3, w4, bias:', w1, w2, w3, w4, bias)

    learning_rate = 0.01
    age = features[:, 0]
    sex = features[:, 1]
    bmi = features[:, 2]
    bp = features[:, 3]

    for i in range(iter_epochs):
        for batch_step in range(0, target.shape[0], batch_size):
            age_batch = age[batch_step:batch_step + batch_size]
            sex_batch = sex[batch_step:batch_step + batch_size]
            bmi_batch = bmi[batch_step:batch_step + batch_size]
            bp_batch = bp[batch_step:batch_step + batch_size]
            target_batch = target[batch_step:batch_step + batch_size]

            bias_update, w1_update, w2_update, w3_update, w4_update = get_update_weights_values_batch(bias, w1, w2, w3, w4, age_batch, sex_batch, bmi_batch, bp_batch, target_batch, learning_rate)

            w1 = w1 - w1_update
            w2 = w2 - w2_update
            w3 = w3 - w3_update
            w4 = w4 - w4_update
            bias = bias - bias_update

            if verbose:
                print('Epoch:', i+1, '/', iter_epochs, 'batch_step:', batch_step)

                predicted = w1 * age + w2 * sex + w3 * bmi + w4 * bp + bias
                diff = target - predicted
                mse_loss = np.mean(np.square(diff))
                print('w1:', w1, 'w2:', w2, 'w3:', w3, 'w4:', w4, 'bias:', bias)

    return w1, w2, w3, w4, bias

In [ ]:
w1, w2, w3, w4, bias = batch_gradient_descent(features, diabetesDF['target'].values, iter_epochs=5000, batch_size=30, verbose=True)
print('##### 최종 w1, w2, w3, w4, bias #######')
print(w1, w2, w3, w4, bias)

스트리밍 출력 내용이 길어서 마지막 5000줄이 삭제되었습니다.
Epoch: 4834 / 5000 batch_step: 180
w1: [50.79977035] w2: [-97.00624565] w3: [758.28529309] w4: [419.35166816] bias: [151.8080109]
Epoch: 4834 / 5000 batch_step: 210
w1: [50.77382137] w2: [-96.99847658] w3: [758.28287039] w4: [419.35054066] bias: [151.94846082]
Epoch: 4834 / 5000 batch_step: 240
w1: [50.77099997] w2: [-96.99361752] w3: [758.30995615] w4: [419.36060887] bias: [151.76350529]
Epoch: 4834 / 5000 batch_step: 270
w1: [50.7922525] w2: [-96.99874547] w3: [758.30884732] w4: [419.37054062] bias: [151.88132597]
Epoch: 4834 / 5000 batch_step: 300
w1: [50.80207268] w2: [-96.99029282] w3: [758.29541559] w4: [419.37118113] bias: [151.87131381]
Epoch: 4834 / 5000 batch_step: 330
w1: [50.80336568] w2: [-96.99074577] w3: [758.30589017] w4: [419.36367468] bias: [152.08181616]
Epoch: 4834 / 5000 batch_step: 360
w1: [50.8002799] w2: [-96.98184939] w3: [758.29923114] w4: [419.36058556] bias: [152.08769518]
Epoch: 4834 / 5000 batch_step: 390
w1: [50.8134288

In [ ]:
predicted = features[:, 0] * w1 + features[:, 1] * w2 + features[:, 2] * w3 + features[:, 3] * w4 + bias
diabetesDF['BATCH_PREDICTED_TARGET'] = predicted
diabetesDF.head(10)

,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6,target,BATCH_PREDICTED_TARGET
0,0.038076,0.050680,0.061696,0.021872,-0.044223,-0.034821,-0.043401,-0.002592,0.019907,-0.017646,151.0,205.062376
1,-0.001882,-0.044642,-0.051474,-0.026328,-0.008449,-0.019163,0.074412,-0.039493,-0.068332,-0.092204,75.0,106.087359
2,0.085299,0.050680,0.044451,-0.005670,-0.045599,-0.034194,-0.032356,-0.002592,0.002861,-0.025930,141.0,182.718261
3,-0.089063,-0.044642,-0.011595,-0.036656,0.012191,0.024991,-0.036038,0.034309,0.022688,-0.009362,206.0,127.796714
4,0.005383,-0.044642,-0.036385,0.021872,0.003935,0.015596,0.008142,-0.002592,-0.031988,-0.046641,135.0,138.151720
5,-0.092695,-0.044642,-0.040696,-0.019442,-0.068991,-0.079288,0.041277,-0.076395,-0.041176,-0.096346,97.0,112.687091
6,-0.045472,0.050680,-0.047163,-0.015999,-0.040096,-0.024800,0.000779,-0.039493,-0.062917,-0.038357,138.0,102.181115
7,0.063504,0.050680,-0.001895,0.066629,0.090620,0.108914,0.022869,0.017703,-0.035816,0.003064,63.0,176.691185
8,0.041708,0.050680,0.061696,-0.040099,-0.013953,0.006202,-0.028674,-0.002592,-0.014960,0.011349,110.0,179.245970
9,-0.070900,-0.044642,0.039062,-0.033213,-0.012577,-0.034508,-0.024993,-0.002592,0.067737,-0.013504,310.0,168.698749


# Mini BATCH GD를 keras로 수행
 - keras는 기본적으로 mini batch gd를 수행

In [ ]:
from tensorflow.keras.layers import Dense
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam

model = Sequential([
    # 단 하나의 units 설정.
    # input_shape는 2차원, 회귀이므로 activation은 설정하지 않음.
    Dense(1, input_shape=(4,), activation=None, kernel_initializer='zeros', bias_initializer='ones')
])
# Adam optimizer를 이용하고 loss 함수는 mse, 성능측정도 mse를 이용하여 학습 수행.
model.compile(optimizer=Adam(learning_rate=0.01), loss='mse', metrics=['mse'])

# keras는 반드시 batch GD를 적용함. batch_size가 none이면 32를 할당.
model.fit(features, diabetesDF['target'].values, batch_size=30, epochs=1000)

Epoch 1/1000
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - loss: 28749.9102 - mse: 28749.9102
Epoch 2/1000
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 28703.8105 - mse: 28703.8105 
Epoch 3/1000
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 28657.6973 - mse: 28657.6973 
Epoch 4/1000
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 28612.2246 - mse: 28612.2246 
Epoch 5/1000
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 28565.7227 - mse: 28565.7227 
Epoch 6/1000
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 28520.4824 - mse: 28520.4824 
Epoch 7/1000
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 28474.7754 - mse: 28474.7754 
Epoch 8/1000
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 28428.8301 - mse: 28428.8301 
Epoch 9/1000
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 28383.5156 - mse: 28383.5156 
Epoch 10/1000
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 28338.4102 - mse: 28338.4102 
Epoch 11/1000
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 28292.0664 - mse: 28292.0664 
Epoch 12

In [ ]:
predicted = model.predict(features)
diabetesDF['KERAS_BATCH_PREDICTED_TARGET'] = predicted
diabetesDF.head(10)

14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step


,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6,target,BATCH_PREDICTED_TARGET,KERAS_BATCH_PREDICTED_TARGET
0,0.038076,0.050680,0.061696,0.021872,-0.044223,-0.034821,-0.043401,-0.002592,0.019907,-0.017646,151.0,205.062376,145.100998
1,-0.001882,-0.044642,-0.051474,-0.026328,-0.008449,-0.019163,0.074412,-0.039493,-0.068332,-0.092204,75.0,106.087359,121.140495
2,0.085299,0.050680,0.044451,-0.005670,-0.045599,-0.034194,-0.032356,-0.002592,0.002861,-0.025930,141.0,182.718261,143.085831
3,-0.089063,-0.044642,-0.011595,-0.036656,0.012191,0.024991,-0.036038,0.034309,0.022688,-0.009362,206.0,127.796714,118.709915
4,0.005383,-0.044642,-0.036385,0.021872,0.003935,0.015596,0.008142,-0.002592,-0.031988,-0.046641,135.0,138.151720,129.221313
5,-0.092695,-0.044642,-0.040696,-0.019442,-0.068991,-0.079288,0.041277,-0.076395,-0.041176,-0.096346,97.0,112.687091,116.782059
6,-0.045472,0.050680,-0.047163,-0.015999,-0.040096,-0.024800,0.000779,-0.039493,-0.062917,-0.038357,138.0,102.181115,120.875191
7,0.063504,0.050680,-0.001895,0.066629,0.090620,0.108914,0.022869,0.017703,-0.035816,0.003064,63.0,176.691185,144.124130
8,0.041708,0.050680,0.061696,-0.040099,-0.013953,0.006202,-0.028674,-0.002592,-0.014960,0.011349,110.0,179.245970,138.100143
9,-0.070900,-0.044642,0.039062,-0.033213,-0.012577,-0.034508,-0.024993,-0.002592,0.067737,-0.013504,310.0,168.698749,126.832031
